# 03 · Process into the final dataset

Reads every kept photo, checks its SHA-1 against the manifest, removes validation and test photos whose
perceptual hash already appears in an earlier split, and writes the final `<split>/<species>/` layout with
`metadata.csv`, `classes.csv`, a processing report and sample grids.

- **Reads:** the raw dataset (01) and the split dataset (02). Set the two handles below if you rebuilt
  them under your own account.
- **Saves:** `<your username>/serpentguard-indian-snakes-processed` (data.zip, which Kaggle unpacks,
  plus metadata.csv, classes.csv, processing_report.json, figures/, README.md).
- **Time:** about 15 minutes of processing plus about 20 minutes of upload on CPU.

Use `class_id` from `classes.csv` for every split: `val/` and `test/` lack some species folders, so
loaders that number classes by folder (plain `ImageFolder`) would mislabel photos.

| # | Notebook | Reads | Saves as dataset |
|---|---|---|---|
| 01 | `01_build_raw_dataset` | iNaturalist API | `<you>/serpentguard-indian-snakes` |
| 02 | `02_clean_and_split` | 01 | `<you>/serpentguard-m2-splits` |
| 03 | `03_process_dataset` | 01 and 02 | `<you>/serpentguard-indian-snakes-processed` |
| 04 | `04_sam_masks` | 01 and 02 | snake masks (per shard) |

In [ ]:
import kagglehub

OWNER = kagglehub.whoami()["username"]
print("running as", OWNER)

RAW_HANDLE = "puneetbajajiitm/serpentguard-indian-snakes"
SPLITS_HANDLE = "puneetbajajiitm/serpentguard-m2-splits"
UPLOAD = True
print("raw:", kagglehub.dataset_download(RAW_HANDLE))
print("splits:", kagglehub.dataset_download(SPLITS_HANDLE))

In [ ]:
import collections
import hashlib
import io
import json
import shutil
import threading
import time
import zipfile
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from scipy.fft import dctn

DATASET_HANDLE = "puneetbajajiitm/serpentguard-indian-snakes-processed"
INPUT_ROOT = Path("/kaggle/input")
RAW_SLUG = "serpentguard-indian-snakes"
SPLITS_SLUG = "serpentguard-m2-splits"
SCRATCH = Path("/kaggle/tmp/serpentguard-processed")
REPORT_DIR = Path("/kaggle/working")
ZIP_NAME = "data.zip"
SPLIT_ORDER = ("train", "val", "test", "test_extra", "unknown_pool")
DEDUP_SPLITS = ("val", "test", "test_extra")
READ_WORKERS = 32
PROGRESS_EVERY = 10000
PHASH_SIZE = 32
PHASH_KEEP = 8
GRID_TILE = 112
GRID_COLUMNS = 16
SAMPLES_PER_SPECIES = 4
SAMPLE_SEED = 7

rows_by_idx: dict = {}

README = """# SerpentGuard Indian snakes, processed for modelling (private, course use only)

IIT Madras BS, DS and AI Lab, Group 5. Milestone 2 processed dataset.

data/ (from data.zip)
  train/<species>/          training photos
  val/<species>/            validation photos
  test/<species>/           test photos: taken in India, uploaded on or after 1 Jan 2022
  test_extra/<species>/     held-out photos from test-fold photographers that are not
                            Indian or were uploaded before 2022 (secondary evaluation)
  unknown_pool/<species>/   species with fewer than 20 clean photos; never trained on,
                            used to test the "unknown" answer
metadata.csv   one row per photo: path, split, species, genus, family, WHO category,
               tier, observation, observer, dates, India flag, licence, attribution, pHash
classes.csv    one row per class with per-split counts and labels. Use its class_id for
               every split: val/ and test/ lack some species folders, so loaders that
               number classes by folder (plain ImageFolder) would mislabel photos
processing_report.json   counts, drops and leakage checks
figures/       sample image grids

Split rule: photographers are grouped (StratifiedGroupKFold, 7 folds, seed 42), so no
photographer, observation or identical image appears in more than one of train, val
and the test fold. Later splits drop photos whose perceptual hash matches an earlier
split. Photo copyright stays with each photographer; see the license and attribution
columns. Use cc_licensed == True for a Creative Commons only subset.
"""


def find_input(slug: str) -> Path:
    for path in INPUT_ROOT.rglob(slug):
        if path.is_dir():
            return path
    raise FileNotFoundError(slug)


def phash(body: bytes) -> str:
    with Image.open(io.BytesIO(body)) as image:
        gray = np.asarray(image.convert("L").resize((PHASH_SIZE, PHASH_SIZE), Image.LANCZOS), dtype=np.float32)
    low = dctn(gray, norm="ortho")[:PHASH_KEEP, :PHASH_KEEP].flatten()
    bits = low > np.median(low[1:])
    return f"{int(''.join('1' if b else '0' for b in bits), 2):016x}"


class ZipSink:
    def __init__(self, path: Path):
        self.zip = zipfile.ZipFile(path, "w", compression=zipfile.ZIP_STORED, allowZip64=True)
        self.lock = threading.Lock()

    def add(self, name: str, body: bytes) -> None:
        with self.lock:
            self.zip.writestr(name, body)

    def close(self) -> None:
        self.zip.close()


def load(row: dict, raw_root: Path) -> dict:
    try:
        body = (raw_root / "images" / row["file"]).read_bytes()
        return {"idx": row["idx"], "body": body, "sha_ok": hashlib.sha1(body).hexdigest() == row["sha1"],
                "phash": phash(body)}
    except Exception as error:
        return {"idx": row["idx"], "body": None, "error": repr(error)[:200]}


def process_split(rows: list, raw_root: Path, sink: ZipSink, seen: set, dedup: bool, out: dict) -> None:
    with ThreadPoolExecutor(READ_WORKERS) as pool:
        for i, result in enumerate(pool.map(lambda r: load(r, raw_root), rows), 1):
            row = rows_by_idx[result["idx"]]
            if result["body"] is None:
                out[result["idx"]] = ("read_error", None, False)
            elif dedup and result["phash"] in seen:
                out[result["idx"]] = ("near_duplicate", result["phash"], result["sha_ok"])
            else:
                sink.add(row["path"], result["body"])
                seen.add(result["phash"])
                out[result["idx"]] = ("kept", result["phash"], result["sha_ok"])
            if i % PROGRESS_EVERY == 0:
                print(time.strftime("%H:%M:%S"), f"{row['split']}: {i}/{len(rows)}", flush=True)


def sample_grid(df: pd.DataFrame, data_dir_zip: zipfile.ZipFile, species: list, per_species: int, title: str, path: Path) -> None:
    rng = np.random.default_rng(SAMPLE_SEED)
    tiles = []
    for name in species:
        pool = df[(df["species"] == name) & (df["split"] == "train")]
        if pool.empty:
            continue
        pick = pool.iloc[rng.choice(len(pool), size=min(per_species, len(pool)), replace=False)]
        for p in pick["path"]:
            with Image.open(io.BytesIO(data_dir_zip.read(p))) as im:
                tiles.append(im.convert("RGB").resize((GRID_TILE, GRID_TILE)))
    columns = per_species if per_species > 1 else GRID_COLUMNS
    rows = -(-len(tiles) // columns)
    canvas = Image.new("RGB", (columns * GRID_TILE, rows * GRID_TILE), "white")
    for i, tile in enumerate(tiles):
        canvas.paste(tile, ((i % columns) * GRID_TILE, (i // columns) * GRID_TILE))
    canvas.save(path, quality=90)
    (path.parent / f"{path.stem}_order.txt").write_text(f"{title}\n" + "\n".join(species))


def main() -> None:
    started = time.time()
    raw_root, splits_root = find_input(RAW_SLUG), find_input(SPLITS_SLUG)
    print("raw:", raw_root, "splits:", splits_root, flush=True)
    df = pd.read_csv(splits_root / "splits.csv", dtype={"obs_id": str, "photo_id": str, "observer_id": str, "sha1": str})
    df["idx"] = range(len(df))
    rows_by_idx.update(df.set_index("idx", drop=False).to_dict("index"))

    SCRATCH.mkdir(parents=True, exist_ok=True)
    sink = ZipSink(SCRATCH / ZIP_NAME)
    outcome, seen = {}, set()
    for split in SPLIT_ORDER:
        part = df[df["split"] == split].to_dict("records")
        print(time.strftime("%H:%M:%S"), f"{split}: {len(part)} photos", flush=True)
        process_split(part, raw_root, sink, seen, split in DEDUP_SPLITS, outcome)
    sink.close()

    df["status"] = df["idx"].map(lambda i: outcome[i][0])
    df["phash"] = df["idx"].map(lambda i: outcome[i][1])
    df["sha1_verified"] = df["idx"].map(lambda i: outcome[i][2])
    kept = df[df["status"] == "kept"].drop(columns=["idx", "status"])
    kept.to_csv(SCRATCH / "metadata.csv", index=False)

    labelled = kept[kept["split"] != "unknown_pool"]
    classes = labelled.pivot_table(index="species", columns="split", values="photo_id", aggfunc="count", fill_value=0)
    classes = classes.reindex(columns=["train", "val", "test", "test_extra"], fill_value=0)
    meta = labelled.drop_duplicates("species").set_index("species")[["genus", "family", "who_category", "tier", "m1_core27", "taxon_id"]]
    classes = classes.join(meta)
    classes["total"] = classes[["train", "val", "test", "test_extra"]].sum(axis=1)
    classes = classes.sort_values("total", ascending=False)
    classes.insert(0, "class_id", range(len(classes)))
    classes.to_csv(SCRATCH / "classes.csv")

    leakage = {}
    block = labelled["split"].replace({"test_extra": "test"})
    for key in ("observer_id", "obs_id", "sha1", "phash"):
        leakage[f"{key}_in_more_than_one_of_train_val_testfold"] = int(labelled.assign(b=block).groupby(key)["b"].nunique().gt(1).sum())
    report = {
        "minutes": round((time.time() - started) / 60, 1),
        "input_rows": len(df),
        "status": df["status"].value_counts().to_dict(),
        "near_duplicates_by_split": df[df["status"] == "near_duplicate"]["split"].value_counts().to_dict(),
        "sha1_mismatches_among_kept": int((~kept["sha1_verified"]).sum()),
        "kept_by_split": kept["split"].value_counts().to_dict(),
        "classes": len(classes),
        "classes_by_tier": classes["tier"].value_counts().to_dict(),
        "leakage": leakage,
        "bytes_zip": (SCRATCH / ZIP_NAME).stat().st_size,
    }
    (SCRATCH / "processing_report.json").write_text(json.dumps(report, indent=1, default=str))
    (SCRATCH / "README.md").write_text(README)
    print(json.dumps(report, indent=1, default=str), flush=True)

    figures = SCRATCH / "figures"
    figures.mkdir(exist_ok=True)
    with zipfile.ZipFile(SCRATCH / ZIP_NAME) as zf:
        sample_grid(kept, zf, list(classes.index), 1, "one training photo per class, classes in classes.csv order", figures / "grid_all_classes.jpg")
        core = list(classes[classes["m1_core27"]].index)
        sample_grid(kept, zf, core, SAMPLES_PER_SPECIES, "M1 core 27 species, 4 training photos each (one row per species)", figures / "grid_core27.jpg")
        danger = list(classes[classes["who_category"] != "not_listed"].index)
        sample_grid(kept, zf, danger, SAMPLES_PER_SPECIES, "WHO Category 1 and 2 species, 4 training photos each", figures / "grid_who_listed.jpg")
    REPORT_DIR.mkdir(parents=True, exist_ok=True)
    for name in ("metadata.csv", "classes.csv", "processing_report.json"):
        shutil.copy(SCRATCH / name, REPORT_DIR / name)
    shutil.copytree(figures, REPORT_DIR / "figures", dirs_exist_ok=True)

    import kagglehub
    kagglehub.dataset_upload(DATASET_HANDLE, str(SCRATCH), version_notes="M2 processed dataset")
    print(time.strftime("%H:%M:%S"), "uploaded", DATASET_HANDLE, flush=True)

In [ ]:
DATASET_HANDLE = f"{OWNER}/serpentguard-indian-snakes-processed"
INPUT_PATHS = {RAW_SLUG: Path(kagglehub.dataset_download(RAW_HANDLE)), SPLITS_SLUG: Path(kagglehub.dataset_download(SPLITS_HANDLE))}
find_input = INPUT_PATHS.__getitem__
if not UPLOAD:
    kagglehub.dataset_upload = lambda *args, **kwargs: print("UPLOAD is False, not saving", args[0])
main()